In [ ]:
import random
from collections import defaultdict
import matplotlib.pyplot as plt

random.seed(0)

# ============================================================
# Cliff Walking Environment
# ============================================================

class CliffWalking:
    """
    Standard cliff walking grid:
      - start = bottom-left
      - goal  = bottom-right
      - cliff = cells in between on the bottom row
    """
    def __init__(self, rows = 4, cols=12):
        self.rows = rows
        self.cols = cols
        self.start = (rows - 1, 0)
        self.goal  = (rows - 1, cols - 1)
        self.cliff = {(rows - 1, c) for c in range(1, cols - 1)} 
        # cliff is all cells in the bottom row except start and goal
        self.actions = ["U", "D", "L", "R"] # up down left right
        self.moves = {
            "U": (-1, 0),
            "D": (1, 0),
            "L": (0, -1),
            "R": ( 0, 1),
        }

    def step(self, state, action):
        """
        Step cost = -1 per move.
        Falling into cliff = -100 and reset to start.
        Reaching goal ends the episode.
        """
        if state == self.goal:
            return state, 0, True

        dr, dc = self.moves[action]
        row = max(0, min(self.rows - 1, state[0] + dr))
        col = max(0, min(self.cols - 1, state[1] + dc))
        next_state = (row, col)

        # cliff: big penalty, reset to start, episode continues
        if next_state in self.cliff:
            return self.start, -100, False

        # goal: terminal
        if next_state == self.goal:
            return next_state, 0, True

        # normal move negative reward so you encourage shorter paths
        return next_state, -1, False


In [ ]:
# ============================================================
# Helpers
# ============================================================

def epsilon_greedy_action(Q, env, state, eps=0.1):
    if random.random() < eps:
        return random.choice(env.actions)

    best_value = max(Q[(state, a)] for a in env.actions)
    best_actions = [a for a in env.actions if Q[(state, a)] == best_value]
    return random.choice(best_actions)

def greedy_action_from_Q(Q, env, state):
    best_value = max(Q[(state, a)] for a in env.actions)
    best_actions = [a for a in env.actions if Q[(state, a)] == best_value]
    return random.choice(best_actions)

def greedy_action_from_V(V, env, state, gamma=1.0):
    best_score = float("-inf")
    best_actions = []

    for a in env.actions:
        s2, r, done = env.step(state, a)
        score = r if done else r + gamma * V[s2]
        if score > best_score:
            best_score = score
            best_actions = [a]
        elif score == best_score:
            best_actions.append(a)

    return random.choice(best_actions)

def path_from_policy(env, policy_fn, max_steps=100):
    s = env.start
    path = [s]
    for _ in range(max_steps):
        if s == env.goal:
            break
        a = policy_fn(s)
        s, r, done = env.step(s, a)
        path.append(s)
        if done:
            break
    return path

def print_policy_grid(env, policy_fn):
    arrow = {"U": "↑", "D": "↓", "L": "←", "R": "→"}
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            s = (r, c)
            if s == env.start:
                row.append("S")
            elif s == env.goal:
                row.append("G")
            elif s in env.cliff:
                row.append("X")
            else:
                row.append(arrow[policy_fn(s)])
        print(" ".join(f"{x:>2}" for x in row))
    print()

def moving_average(values, window=100):
    if len(values) < window:
        return values[:]
    out = []
    s = sum(values[:window])
    out.append(s / window)
    for i in range(window, len(values)):
        s += values[i] - values[i - window]
        out.append(s / window)
    return out


In [ ]:
# ============================================================
# 1) TD(0): State-value prediction under a random policy
# ============================================================

def td0_policy_evaluation(env, episodes=5000, alpha=0.1, gamma=1.0, max_steps=500):
    """
    Evaluates a RANDOM policy.
    TD(0) learns V(s), not a control policy.
    """
    V = defaultdict(float)
    returns = []
    start_values = []

    for ep in range(episodes):
        s = env.start
        total_reward = 0

        for _ in range(max_steps):
            a = random.choice(env.actions)   # random policy
            s2, r, done = env.step(s, a)

            # TD(0) update
            V[s] = V[s] + alpha * (r + gamma * V[s2] - V[s])

            total_reward += r
            s = s2

            if done:
                break

        returns.append(total_reward)
        start_values.append(V[env.start])

        if (ep + 1) % 1000 == 0:
            print(f"TD(0) episode {ep+1}: V(start) = {V[env.start]:.3f}")

    return V, returns, start_values


# ============================================================
# 2) SARSA: On-policy control
# ============================================================

def sarsa(env, episodes=5000, alpha=0.5, gamma=1.0, eps=0.1, max_steps=500):
    """
    On-policy control.
    Learns Q(s,a) using the actual next epsilon-greedy action.
    """
    Q = defaultdict(float)
    returns = []

    for ep in range(episodes):
        s = env.start
        a = epsilon_greedy_action(Q, env, s, eps)
        total_reward = 0

        for _ in range(max_steps):
            s2, r, done = env.step(s, a)
            total_reward += r

            if done:
                target = r
                Q[(s, a)] += alpha * (target - Q[(s, a)])
                break
            else:
                a2 = epsilon_greedy_action(Q, env, s2, eps)
                target = r + gamma * Q[(s2, a2)]
                Q[(s, a)] += alpha * (target - Q[(s, a)])
                s, a = s2, a2

        returns.append(total_reward)

        if (ep + 1) % 1000 == 0:
            policy = lambda st: greedy_action_from_Q(Q, env, st)
            path = path_from_policy(env, policy)
            print(f"SARSA episode {ep+1}: return(last)={total_reward}, greedy path len={len(path)}")

    return Q, returns


# ============================================================
# 3) Q-learning: Off-policy control
# ============================================================

def q_learning(env, episodes=5000, alpha=0.5, gamma=1.0, eps=0.1, max_steps=500):
    """
    Off-policy control.
    Learns Q(s,a) using the greedy best next action in the update.
    """
    Q = defaultdict(float)
    returns = []

    for ep in range(episodes):
        s = env.start
        total_reward = 0

        for _ in range(max_steps):
            a = epsilon_greedy_action(Q, env, s, eps)
            s2, r, done = env.step(s, a)
            total_reward += r

            if done:
                target = r
            else:
                target = r + gamma * max(Q[(s2, a2)] for a2 in env.actions)

            Q[(s, a)] += alpha * (target - Q[(s, a)])

            if done:
                break

            s = s2

        returns.append(total_reward)

        if (ep + 1) % 1000 == 0:
            policy = lambda st: greedy_action_from_Q(Q, env, st)
            path = path_from_policy(env, policy)
            print(f"Q-learning episode {ep+1}: return(last)={total_reward}, greedy path len={len(path)}")

    return Q, returns


In [ ]:
# ============================================================
# Run all three
# ============================================================

env = CliffWalking()

print("Training TD(0) (policy evaluation under random policy)...")
V_td, td_returns, td_start_values = td0_policy_evaluation(env, episodes=5000, alpha=0.1, gamma=1.0)

print("\nTraining SARSA...")
Q_sarsa, sarsa_returns = sarsa(env, episodes=5000, alpha=0.5, gamma=1.0, eps=0.1)

print("\nTraining Q-learning...")
Q_q, q_returns = q_learning(env, episodes=5000, alpha=0.5, gamma=1.0, eps=0.1)

# ============================================================
# Show learned policies / paths
# ============================================================

print("\n--- TD(0) derived greedy policy from V ---")
td_policy = lambda s: greedy_action_from_V(V_td, env, s, gamma=1.0)
print_policy_grid(env, td_policy)
print("TD greedy path:", path_from_policy(env, td_policy))

print("--- SARSA greedy policy from Q ---")
sarsa_policy = lambda s: greedy_action_from_Q(Q_sarsa, env, s)
print_policy_grid(env, sarsa_policy)
print("SARSA greedy path:", path_from_policy(env, sarsa_policy))

print("--- Q-learning greedy policy from Q ---")
q_policy = lambda s: greedy_action_from_Q(Q_q, env, s)
print_policy_grid(env, q_policy)
print("Q-learning greedy path:", path_from_policy(env, q_policy))

# ============================================================
# Plot learning curves
# ============================================================

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(moving_average(td_returns, 100), label="TD(0) random-policy return")
plt.plot(moving_average(sarsa_returns, 100), label="SARSA return")
plt.plot(moving_average(q_returns, 100), label="Q-learning return")
plt.title("Episode Return (moving average)")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(td_start_values, label="TD(0) V(start)")
plt.title("TD(0) Value of Start State")
plt.xlabel("Episode")
plt.ylabel("V(start)")
plt.legend()

plt.tight_layout()
plt.show()